# Real-World Project: Personal Knowledge Assistant

In this project, we'll build a server that helps an AI interact with your local markdown notes.

## 1. Architecture Diagram

Below is the high-level architecture of our MCP Knowledge Assistant:

```mermaid
graph TD
    User[User] <--> Client[MCP Client / Claude Desktop]
    Client <--> Protocol[JSON-RPC over Stdio]
    Protocol <--> Server[Knowledge Assistant Server]
    
    subgraph Server Capabilities
        Tools[Tools: list_notes, read_note]
        Resources[Resources: notes://index]
        Prompts[Prompts: summarize_note]
    end
    
    Server <--> FS[(Local File System /my_notes)]
```

*Note: If your viewer doesn't support Mermaid, the flow is: User -> Claude -> JSON-RPC -> Python Server -> File System.*

## 2. Design Decisions

- **Transport**: We use `stdio` because it's local, fast, and secure for personal data.
- **Framework**: `FastMCP` is used to minimize boilerplate and handle JSON-RPC automatically.
- **Security**: We implement a simple check in `read_note` to prevent directory traversal attacks (e.g., trying to read `/etc/passwd`).

## 3. Implementation Details

The server exposes three main things:
1.  **Tools**: Functions the LLM can *do* (Action-oriented).
2.  **Resources**: Data the LLM can *see* (Context-oriented).
3.  **Prompts**: Instruction templates the LLM can *follow* (Workflow-oriented).

## 4. Complete Code

You can find the full source code in `sample_server.py`. Here is a snippet of the tool implementation:

In [ ]:
import os
from fastmcp import FastMCP

mcp = FastMCP("KnowledgeAssistant")
NOTES_DIR = "my_notes"

@mcp.tool()
def read_note(filename: str) -> str:
    """Reads the content of a specific note."""
    path = os.path.join(NOTES_DIR, filename)
    # Security check
    if ".." in filename or filename.startswith("/"):
        return "Error: Access denied."
    with open(path, 'r') as f:
        return f.read()


---
**Summary**: This project demonstrates how a single Python file can transform a collection of local documents into a powerful, interactive knowledge base for an AI agent.